## All the imports
import data set and all libraries that will be used later


In [ ]:
import pandas as pd

from matplotlib import pyplot as plt 
%matplotlib inline

import numpy as np
import seaborn as sb #heatmap

import os

import random
random.seed(67)
np.random.seed(67)


from sklearn.preprocessing import StandardScaler

# models
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, KFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# nn
import tensorflow as tf
tf.random.set_seed(67)
from tensorflow.keras.callbacks import EarlyStopping

# evaluation
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


# path to the file to read
pcos_file_path = 'data\data without infertility _final.csv'

pcos_data = pd.read_csv(pcos_file_path) 

## Data info
seeing some statistics about the data

In [ ]:
print("Columns BEFORE stripping:")
print(list(pcos_data.columns))

pcos_data.columns = pcos_data.columns.str.strip()

print("\nColumns AFTER stripping:")
print(list(pcos_data.columns))


In [ ]:
#to see empty data ; in our case it is col 42
sb.heatmap(pcos_data.isnull())

## Cleaning data
We have to remove null values and all values should be numerical

In [ ]:
#dropping col that has all null values, and then id cols cause they are not needed for the clasification
#axis=1 is for col axis=0 is for rows
pcos_data.drop(["Unnamed: 42", "Sl. No", "Patient File No."], axis=1, inplace=True)

In [ ]:
sb.heatmap(pcos_data.isnull())

In [ ]:
pcos_data.isnull().sum()

In [ ]:
pcos_data["Cycle(R/I)"] = pcos_data["Cycle(R/I)"].map({
    2: 0,  # Regular
    4: 1   # Irregular
})
# Channging the value of Irregular/regular cycles

### Setting features

In [ ]:
hormone_features = [
    "FSH(mIU/mL)",
    "LH(mIU/mL)",
    "FSH/LH",
    "AMH(ng/mL)",
    "TSH (mIU/L)",
    "PRL(ng/mL)",
    "PRG(ng/mL)"  
]

patient_observable_features = [
    "Age (yrs)",
    "Weight (Kg)",
    "Height(Cm)",
    "Pregnant(Y/N)",
    "No. of aborptions",
    "Cycle(R/I)",             # Regular/Irregular periods
    "Cycle length(days)",     # length of period
    "Weight gain(Y/N)",
    "hair growth(Y/N)",
    "Skin darkening (Y/N)",
    "Hair loss(Y/N)",
    "Pimples(Y/N)",
    "Fast food (Y/N)",
    "Reg.Exercise(Y/N)"       # lifestyle activity
]

doctor_observable_features = [
    # Patient-reported symptoms
    "Cycle(R/I)",
    "Cycle length(days)",
    "Weight gain(Y/N)",
    "hair growth(Y/N)",
    "Skin darkening (Y/N)",
    "Hair loss(Y/N)",
    "Pimples(Y/N)",
    "Fast food (Y/N)",
    "Reg.Exercise(Y/N)",
    
    # Doctor-observed measurements
    "Age (yrs)",
    "Weight (Kg)",
    "Height(Cm)",
    "BMI",
    "Pulse rate(bpm)",
    "RR (breaths/min)",
    "Hip(inch)",
    "Waist(inch)",
    "Waist:Hip Ratio",
    "BP _Systolic (mmHg)",
    "BP _Diastolic (mmHg)",
    
    # Ultrasound / gynecologist measurements
    "Follicle No. (L)",
    "Follicle No. (R)",
    "Avg. F size (L) (mm)",
    "Avg. F size (R) (mm)",
    "Endometrium (mm)"
]


In [ ]:
# all features except ids and target
# all_features = pcos_data.drop(columns=['PCOS (Y/N)'], axis=1)

all_features = list(pcos_data.drop(columns=['PCOS (Y/N)']).columns)

print(all_features)


Assigning what features we wanna use

In [ ]:
pcos_features = doctor_observable_features

Last cleaning data procces is removing null values from feature list

In [ ]:
# dropna drops missing values (think of na as "not available")
pcos_data = pcos_data.dropna(
    subset=pcos_features + ["PCOS (Y/N)"]
)
#drop a row only if one of features or pcos label is missing

In [ ]:
pcos_data[pcos_features].isnull().sum()


## Separating predictor and targets

In [ ]:
# By convention, this data (features) is called X.
X = pcos_data[pcos_features]

# want to predict if paitent has pcos or no
y = pcos_data["PCOS (Y/N)"]

In [ ]:
X.head()

In [ ]:
y.value_counts()

In [ ]:
y.value_counts().plot.pie(autopct='%.2f')

## Spliting data into train and test

In [ ]:
#the function train_test_split returns 4 diff values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=67)
# if you ever want to repeat the same split, you have to repeat random_state num
# apperently Keras does val split for monitoring
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=67)



## Scale the data - normalize 

In [ ]:
#create a scaler object so that we can normalize the data
scaler = StandardScaler()

# fit the scaler to the data and then transform the data
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


X_val = scaler.transform(X_val)


# Logistic Regression

## Train the model

In [ ]:
logistic_model = LogisticRegression(
    max_iter=500,
    random_state=67
)
# missing pcos patients is worse than misclassifying a non PCOS-patient
# so the clas 1 weights twice more than class 0

# https://www.geeksforgeeks.org/machine-learning/how-to-optimize-logistic-regression-performance/
param_grid_LR = {
    "C": [0.1, 1, 2],
    "class_weight": [
        None,
        {0:1, 1:1.5},
        {0:1, 1:2},
    ]
}

grid_search_LR = GridSearchCV(estimator=logistic_model, param_grid=param_grid_LR, cv=10, scoring="recall", n_jobs=-1,verbose=2)

grid_search_LR.fit(X_train, y_train)

best_lr = grid_search_LR.best_estimator_

print("Best params:", grid_search_LR.best_params_)
print("Best CV recall:", grid_search_LR.best_score_)



## Evaluation

In [ ]:
y_probs = best_lr.predict_proba(X_test)[:, 1]
y_pred_lr = (y_probs>= 0.4).astype(int)

accuracy = accuracy_score(y_test, y_pred_lr)

print(f"Test Accuracy: {accuracy:.4f}")

print("\nClassification Report:\n", classification_report(y_test, y_pred_lr))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_lr))

# Random Forest Classifier

## Train the model


In [ ]:
rf_model = RandomForestClassifier(random_state=67)

# https://www.geeksforgeeks.org/machine-learning/random-forest-hyperparameter-tuning-in-python/
param_grid_RF = {
    'n_estimators': [100, 200, 300],  
    'max_depth': [None, 5, 10, 15],       
    'max_features':["sqrt", "log2"],    
    'min_samples_leaf': [1, 2, 4],  
     "class_weight": [
        None,
        {0:1, 1:1.5},
        {0:1, 1:2},
    ]              
}
#dictionary with parameters names (str) as keys and lists of parameter settings


#scoring strategy to evaluate the performance of the cross-validated model on the test set. I choose recall because i want least number of false negatives
#n_jobs -> -1 means using all processors
#verbose : Controls the verbosity: the higher, the more messages. ( опширност )
grid_search = GridSearchCV(rf_model, param_grid_RF, cv=10, scoring='recall', n_jobs=-1, verbose=2)
grid_search.fit(X_train, y_train)

best_rf = grid_search.best_estimator_

print("Best parameters: ", grid_search.best_params_)
print("Best score: ", grid_search.best_score_)

## Evaluation

In [ ]:
y_predictions_rf = best_rf.predict(X_test)

test_accuracy = accuracy_score(y_test, y_predictions_rf)
print(f"Test Accuracy: {test_accuracy:.4f}")

print("\nClassification Report:\n", classification_report(y_test, y_predictions_rf))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_predictions_rf))

In [ ]:
feature_importances = best_rf.feature_importances_
feature_names = pcos_features
sorted_indices = np.argsort(feature_importances)[::-1]

top_k = 10
indices = sorted_indices[:top_k]

plt.figure(figsize=(8,5))
plt.barh(
    np.array(feature_names)[indices][::-1],
    feature_importances[indices][::-1]
)
plt.xlabel("Feature Importance")
plt.title("Top 10 Random Forest Features")
plt.tight_layout()
plt.show()


# Artificial Neural Network

In [ ]:
def plot_loss(history):
    plt.plot(history.history['loss'], label='train_loss')
    plt.plot(history.history['val_loss'], label='val_loss')
    plt.xlabel('Epoch')
    plt.ylabel('Binary Crossentropy Loss')
    plt.legend()
    plt.grid(True)
    plt.show()

def plot_accuracy(history):
    plt.plot(history.history['accuracy'], label='train_accuracy')
    plt.plot(history.history['val_accuracy'], label='val_accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)
    plt.show()

def plot_recall(history):
    plt.plot(history.history['recall'], label='train_recall')
    plt.plot(history.history['val_recall'], label='val_recall')
    plt.xlabel('Epoch')
    plt.ylabel('Recall')
    plt.legend()
    plt.grid(True)
    plt.show()

## Train the model

In [ ]:
def train_model(X_train, y_train, num_nodes, dropout_prob, lr, batch_size, epochs):
    nn_model = tf.keras.Sequential([
        tf.keras.layers.Dense(num_nodes, activation='relu', input_shape=(X_train.shape[1],)),
        tf.keras.layers.Dropout(dropout_prob), #there to prevent overfitting
        tf.keras.layers.Dense(num_nodes, activation='relu'),
        tf.keras.layers.Dropout(dropout_prob),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])

    nn_model.compile(                       # default learning rate for Adam
        optimizer=tf.keras.optimizers.Adam(lr), 
        loss='binary_crossentropy', 
        metrics=['accuracy','recall']
    )

    #nn_model.summary()

    class_weight = {0: 1, 1: 2} 
    history = nn_model.fit(
        X_train, 
        y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        #callbacks=[early_stop], https://keras.io/api/callbacks/early_stopping/
        verbose=0,
        class_weight = class_weight
    )

    return nn_model, history

In [ ]:
best_recall = -1
best_recall_model = None
best_model_history = None
epochs=65
for num_nodes in [8,16,32]:
    for dropout_prob in [0, 0.2]: # 20% of neurons are randomly set to zero at each training step
        for lr in [0.005, 0.001]:
            for batch_size in [16,32,64]:
                model, history = train_model(X_train, y_train, num_nodes, dropout_prob, lr, batch_size, epochs)
                #plot_loss(history)
                #plot_accuracy(history)
                #plot_recall(history)
                val_loss, val_accuracy, val_recall = model.evaluate(X_val, y_val, verbose=0)
                if val_recall > best_recall:  # maximize recall
                    best_recall = val_recall
                    best_recall_model = model
                    best_model_history = history


In [ ]:
print(len(history.history['loss']))

## Evaluation

In [ ]:
plot_loss(best_model_history)
plot_accuracy(best_model_history)
plot_recall(best_model_history)

In [ ]:
y_pred_nn = best_recall_model.predict(X_test)
y_pred_nn = (y_pred_nn > 0.43).astype(int)

In [ ]:
test_accuracy = accuracy_score(y_test, y_pred_nn)
print(f"Test Accuracy: {test_accuracy:.4f}")

print("\nClassification Report:\n", classification_report(y_test, y_pred_nn))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_nn))

# Comparison of models

had a help of ai with this diagrams

In [ ]:
models = {
    "Logistic Regression": y_pred_lr,
    "Random Forest": y_predictions_rf,
    "ANN": y_pred_nn
}

results = []

for name, y_pred in models.items():
    report = classification_report(y_test, y_pred, output_dict=True)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Recall_1": report["1"]["recall"],
        "Precision_1": report["1"]["precision"],
        "F1_1": report["1"]["f1-score"]
    })

results_df = pd.DataFrame(results)
print(results_df)

In [ ]:
model_names = []
fp_values = []
fn_values = []

for name, y_pred in models.items():
    cm = confusion_matrix(y_test, y_pred)

    fp = cm[0, 1]
    fn = cm[1, 0]

    model_names.append(name)
    fp_values.append(fp)
    fn_values.append(fn)

x = np.arange(len(model_names))
width = 0.35

plt.figure(figsize=(8,5))

# Bars
bars_fp = plt.bar(x - width/2, fp_values, width, label="False Positives")
bars_fn = plt.bar(x + width/2, fn_values, width, label="False Negatives", color="red")

plt.xticks(x, model_names, rotation=20)
plt.ylabel("Count")
plt.title("False Positives vs False Negatives")
plt.legend()

# Y-axis ticks by 1
plt.yticks(np.arange(0, max(fp_values + fn_values) + 2, 1))  # +2 to give space for label

# Add numbers on top of each bar
for bar in bars_fp:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.1, int(yval), ha='center', va='bottom')

for bar in bars_fn:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.1, int(yval), ha='center', va='bottom')

plt.show()

## logging

In [ ]:
log_path = r"logs\pcos_model_performance.csv"
os.makedirs(os.path.dirname(log_path), exist_ok=True)  

feature_sets = {
    "hormone_features": hormone_features,
    "patient_observable_features": patient_observable_features,
    "doctor_observable_features": doctor_observable_features,
    "all_features": all_features
}

def get_feature_set_label(features, feature_sets):
    for label, columns in feature_sets.items():
        if set(features) == set(columns):
            return label
    return "custom_features"

columns = [
    "Feature_Set", "Model", "Accuracy", "Recall_1", "Precision_1", "F1_1", 
    "TN", "FP", "FN", "TP"
]

log_df = pd.DataFrame(columns=columns)

models = {
    "Logistic Regression": y_pred_lr,
    "Random Forest": y_predictions_rf,
    "ANN": y_pred_nn
}

feature_set_label = get_feature_set_label(pcos_features, feature_sets)

for model_name, y_pred in models.items():
    report = classification_report(y_test, y_pred, output_dict=True)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    new_entry = {
        "Feature_Set": feature_set_label,
        "Model": model_name,
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "Recall_1": round(report["1"]["recall"], 4),
        "Precision_1": round(report["1"]["precision"], 4),
        "F1_1": round(report["1"]["f1-score"], 4),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    }
    
    log_df = pd.concat([log_df, pd.DataFrame([new_entry])], ignore_index=True)


if os.path.exists(log_path):
    log_df.to_csv(log_path, mode='a', header=False, index=False)
else:
    log_df.to_csv(log_path, index=False)
    
print(f"Logged performance for '{feature_set_label}' to '{log_path}'")